In [1]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal
from scipy.signal import butter, filtfilt, find_peaks
from scipy.interpolate import CubicSpline


# =========================================================
# PATH
# =========================================================
ROOT = Path(r"D:\mmwave-heart-rate-monitoring-demo")

ALIGN_DIR = ROOT / "results" / "alignment"
FILTER_DIR = ROOT / "results" / "filter_v2"
DATA_DIR = ROOT / "data"

FILTER_DIR.mkdir(parents=True, exist_ok=True)

COND_NAMES = [
    "AWR_steady",
    "AWR_unsteady",
    "IWR_steady",
    "IWR_unsteady",
]

# =========================================================
# PARAM
# =========================================================
DUR_ECG = 30.0
HR_MIN = 40
HR_MAX = 180

fmin_hz = HR_MIN / 60.0
fmax_hz = HR_MAX / 60.0

PSD_NPERSEG_SEC = 8.0
PSD_OVERLAP = 0.5
PSD_NFFT_FACTOR = 8

ABP_BW_HZ = 0.35
ROS_WIN_SEC = 1.0
ROS_NSIG = 3.0
LEN_WIN_SEC = 1.0

BASE_FILTERS = ["ABP", "EMD", "ROS", "LEN"]


# =========================================================
# HELPERS
# =========================================================
def sample_key(name):
    m = re.search(r"sample_(\d+)", str(name))
    return int(m.group(1)) if m else 999999


def normalize_pm1(x):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmedian(x)
    m = np.nanmax(np.abs(x))
    if (not np.isfinite(m)) or m == 0:
        return x
    return x / m


def estimate_fs(t):
    t = np.asarray(t, dtype=float)
    dt = np.diff(t)
    dt = dt[(dt > 0) & np.isfinite(dt)]
    if len(dt) == 0:
        raise ValueError("cannot estimate fs from time")
    return 1.0 / np.median(dt)


def parse_seg(seg):
    m = re.match(r"\[(.*),(.*)\]", str(seg))
    if not m:
        return np.nan, np.nan
    return float(m.group(1)), float(m.group(2))


def moving_avg(x, w):
    x = np.asarray(x, dtype=float)
    w = int(max(1, w))
    if w <= 1:
        return x.copy()
    ker = np.ones(w, dtype=float) / float(w)
    return np.convolve(x, ker, mode="same")


def butter_bandpass(x, fs, low, high, order=2):
    x = np.asarray(x, dtype=float)

    if len(x) < max(10, order * 3):
        return x.copy()

    nyq = fs / 2.0
    low = max(float(low), 1e-6)
    high = min(float(high), nyq - 1e-3)

    if not np.isfinite(low) or not np.isfinite(high):
        return x.copy()
    if low >= high:
        return x.copy()

    wn = [low / nyq, high / nyq]
    if wn[0] <= 0:
        wn[0] = 1e-4
    if wn[1] >= 1:
        wn[1] = 0.999

    if wn[0] >= wn[1]:
        return x.copy()

    try:
        b, a = butter(order, wn, btype="band")
        y = filtfilt(b, a, x)
        return y
    except Exception:
        return x.copy()


def _welch_psd(x, fs, nperseg_sec=8.0, overlap=0.5, nfft_factor=8):
    x = np.asarray(x, dtype=float)

    if len(x) < 16 or not np.isfinite(fs) or fs <= 0:
        return None, None

    nperseg = int(round(nperseg_sec * fs))
    nperseg = max(16, min(len(x), nperseg))

    noverlap = int(round(nperseg * overlap))
    noverlap = min(noverlap, nperseg - 1)

    nfft = 1
    target_nfft = max(nperseg, int(nfft_factor * nperseg))
    while nfft < target_nfft:
        nfft *= 2

    try:
        f, Pxx = signal.welch(
            x,
            fs=fs,
            window="hann",
            nperseg=nperseg,
            noverlap=noverlap,
            nfft=nfft,
            detrend="constant",
            scaling="density",
        )
        return f, Pxx
    except Exception:
        return None, None


def estimate_hr_bpm_welch(x, fs, fmin=0.8, fmax=3.0):
    x = np.asarray(x, dtype=float)

    f, Pxx = _welch_psd(
        x, fs,
        nperseg_sec=PSD_NPERSEG_SEC,
        overlap=PSD_OVERLAP,
        nfft_factor=PSD_NFFT_FACTOR
    )
    if f is None:
        return np.nan

    band = (f >= float(fmin)) & (f <= float(fmax))
    if not np.any(band):
        return np.nan

    fb = f[band]
    pb = Pxx[band]

    if len(pb) < 3 or np.all(~np.isfinite(pb)):
        return np.nan

    idx = int(np.nanargmax(pb))
    return float(fb[idx] * 60.0)


def estimate_hr_fft_from_signal(t, x):
    fs = estimate_fs(t)
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    return estimate_hr_bpm_welch(x, fs, fmin=fmin_hz, fmax=fmax_hz)

In [2]:
# =========================================================
# FILTERS
# =========================================================
def filter_abp(x, fs, fmin=0.8, fmax=3.0, bw=0.35):
    x = np.asarray(x, dtype=float)
    nyq = fs / 2.0
    fmax_eff = min(float(fmax), nyq - 0.01)

    x0 = butter_bandpass(x, fs, fmin, fmax_eff, order=2)
    if len(x0) < 16:
        return x0

    hr0 = estimate_hr_bpm_welch(x0, fs, fmin=fmin, fmax=fmax_eff)
    if not np.isfinite(hr0):
        return x0

    f0 = float(hr0) / 60.0
    lo = max(float(fmin), f0 - float(bw))
    hi = min(fmax_eff, f0 + float(bw))

    return butter_bandpass(x, fs, lo, hi, order=2)


def emd_decompose(x, max_imfs=6, max_sift=12, stop_sd=0.05):
    x = np.asarray(x, dtype=float)
    N = len(x)
    imfs = []
    r = x.copy()

    for _ in range(max_imfs):
        h = r.copy()

        for _s in range(max_sift):
            max_idx, _ = find_peaks(h)
            min_idx, _ = find_peaks(-h)

            if (len(max_idx) + len(min_idx)) < 4:
                break

            max_idx = np.r_[0, max_idx, N - 1]
            min_idx = np.r_[0, min_idx, N - 1]

            try:
                upper = CubicSpline(max_idx, h[max_idx])(np.arange(N))
                lower = CubicSpline(min_idx, h[min_idx])(np.arange(N))
            except Exception:
                break

            m = (upper + lower) / 2.0
            h1 = h - m

            sd = np.linalg.norm(h - h1) / (np.linalg.norm(h) + 1e-12)
            h = h1

            if sd < stop_sd:
                break

        imfs.append(h)
        r = r - h

        if np.std(r) < 1e-3 * (np.std(x) + 1e-12):
            break

    return imfs, r


def filter_emd_pick_heart_imf(x, fs, fmin=0.8, fmax=3.0):
    x = np.asarray(x, dtype=float)
    nyq = fs / 2.0
    fmax_eff = min(float(fmax), nyq - 0.01)

    x0 = x - moving_avg(x, int(round(fs * 2.0)))
    x0 = butter_bandpass(x0, fs, fmin, fmax_eff, order=2)

    imfs, _ = emd_decompose(x0, max_imfs=6, max_sift=12, stop_sd=0.05)
    if not imfs:
        return x0

    best = None
    best_pow = -1.0

    for imf in imfs:
        if len(imf) < 16:
            continue

        f, Pxx = _welch_psd(
            imf, fs,
            nperseg_sec=PSD_NPERSEG_SEC,
            overlap=PSD_OVERLAP,
            nfft_factor=PSD_NFFT_FACTOR
        )
        if f is None:
            continue

        band = (f >= float(fmin)) & (f <= fmax_eff)
        if not np.any(band):
            continue

        p = float(np.sum(Pxx[band]))
        if p > best_pow:
            best_pow = p
            best = imf

    if best is None:
        return x0

    return butter_bandpass(best, fs, fmin, fmax_eff, order=2)


def hampel_filter(x, fs, win_sec=1.0, n_sig=3.0):
    x = np.asarray(x, dtype=float).copy()

    w = int(max(3, round(win_sec * fs)))
    if w % 2 == 0:
        w += 1
    k = w // 2

    for i in range(len(x)):
        s = max(0, i - k)
        e = min(len(x), i + k + 1)
        seg = x[s:e]

        med = np.median(seg)
        mad = np.median(np.abs(seg - med)) + 1e-12
        thr = n_sig * 1.4826 * mad

        if abs(x[i] - med) > thr:
            x[i] = med

    return x


def filter_ros(x, fs, fmin=0.8, fmax=3.0, win_sec=1.0, n_sig=3.0):
    x = np.asarray(x, dtype=float)
    nyq = fs / 2.0
    fmax_eff = min(float(fmax), nyq - 0.01)

    x1 = hampel_filter(x, fs, win_sec=win_sec, n_sig=n_sig)
    return butter_bandpass(x1, fs, fmin, fmax_eff, order=2)


def filter_len(x, fs, fmin=0.8, fmax=3.0, win_sec=1.0):
    x = np.asarray(x, dtype=float)
    nyq = fs / 2.0
    fmax_eff = min(float(fmax), nyq - 0.01)

    w = int(max(3, round(win_sec * fs)))
    e = np.sqrt(moving_avg(x**2, w) + 1e-12)
    x1 = x / (e + 1e-12)

    return butter_bandpass(x1, fs, fmin, fmax_eff, order=2)


def build_filters(fs_u):
    return {
        "ABP": lambda x: filter_abp(x, fs_u, fmin=fmin_hz, fmax=fmax_hz, bw=ABP_BW_HZ),
        "EMD": lambda x: filter_emd_pick_heart_imf(x, fs_u, fmin=fmin_hz, fmax=fmax_hz),
        "ROS": lambda x: filter_ros(x, fs_u, fmin=fmin_hz, fmax=fmax_hz, win_sec=ROS_WIN_SEC, n_sig=ROS_NSIG),
        "LEN": lambda x: filter_len(x, fs_u, fmin=fmin_hz, fmax=fmax_hz, win_sec=LEN_WIN_SEC),
    }


def apply_filters(t, x, combo):
    y = np.asarray(x, dtype=float).copy()
    fs = estimate_fs(t)
    filters = build_filters(fs)

    for f in combo:
        y = filters[f](y)

    return y


# =========================================================
# FILTER COMBOS
# =========================================================
FILTER_COMBOS = [
    ("raw", ()),

    ("ABP", ("ABP",)),
    ("EMD", ("EMD",)),
    ("ROS", ("ROS",)),
    ("LEN", ("LEN",)),

    ("ABP→EMD", ("ABP", "EMD")),
    ("ABP→ROS", ("ABP", "ROS")),
    ("ABP→LEN", ("ABP", "LEN")),

    ("EMD→ABP", ("EMD", "ABP")),
    ("EMD→ROS", ("EMD", "ROS")),
    ("EMD→LEN", ("EMD", "LEN")),

    ("ROS→ABP", ("ROS", "ABP")),
    ("ROS→EMD", ("ROS", "EMD")),
    ("ROS→LEN", ("ROS", "LEN")),

    ("LEN→ABP", ("LEN", "ABP")),
    ("LEN→EMD", ("LEN", "EMD")),
    ("LEN→ROS", ("LEN", "ROS")),
]

print("total filters =", len(FILTER_COMBOS))
for item in FILTER_COMBOS:
    print(item)

total filters = 17
('raw', ())
('ABP', ('ABP',))
('EMD', ('EMD',))
('ROS', ('ROS',))
('LEN', ('LEN',))
('ABP→EMD', ('ABP', 'EMD'))
('ABP→ROS', ('ABP', 'ROS'))
('ABP→LEN', ('ABP', 'LEN'))
('EMD→ABP', ('EMD', 'ABP'))
('EMD→ROS', ('EMD', 'ROS'))
('EMD→LEN', ('EMD', 'LEN'))
('ROS→ABP', ('ROS', 'ABP'))
('ROS→EMD', ('ROS', 'EMD'))
('ROS→LEN', ('ROS', 'LEN'))
('LEN→ABP', ('LEN', 'ABP'))
('LEN→EMD', ('LEN', 'EMD'))
('LEN→ROS', ('LEN', 'ROS'))


In [3]:
tables = {}

for cond in COND_NAMES:
    df = pd.read_csv(ALIGN_DIR / f"{cond}.csv")

    df = df[[
        "sample_name",
        "ECG_HR",
        "seg_drscc",
        "mmw_HR_drscc"
    ]].copy()

    df["start"], df["end"] = zip(*df["seg_drscc"].map(parse_seg))
    df = df.sort_values("sample_name", key=lambda s: s.map(sample_key))

    out = df.copy()

    for name, _ in FILTER_COMBOS:
        out[name + "_HR"] = np.nan

    tables[cond] = out
    out.to_csv(FILTER_DIR / f"{cond}.csv", index=False)

    print(cond, "table created")

AWR_steady table created
AWR_unsteady table created
IWR_steady table created
IWR_unsteady table created


In [4]:
for cond in COND_NAMES:
    df = tables[cond]

    print("\n==============================")
    print("Processing:", cond)
    print("samples:", len(df))
    print("==============================")

    for i, row in df.iterrows():
        sample = row["sample_name"]
        start = row["start"]
        end = row["end"]

        print(f"[{i+1}/{len(df)}] {sample}")

        mmw_csv = DATA_DIR / cond / sample / "waveform.csv"

        if not mmw_csv.exists():
            print("  waveform missing")
            continue

        try:
            raw = pd.read_csv(mmw_csv, comment="#", engine="python")
        except Exception:
            print("  read csv failed")
            continue

        raw = raw.dropna()

        if "time" not in raw.columns:
            print("  missing time column")
            continue

        t = raw["time"].values
        x = raw.iloc[:, 1].values

        mask = (t >= start) & (t < end)
        t = t[mask]
        x = x[mask]

        if len(t) < 10:
            print("  too short")
            continue

        t = t - t[0]

        signals = {}
        hrs = {}

        for name, combo in FILTER_COMBOS:
            try:
                if name == "raw":
                    y = np.asarray(x, dtype=float).copy()
                else:
                    y = apply_filters(t, x, combo)

                hr = estimate_hr_fft_from_signal(t, y)

            except Exception as e:
                print("  filter error:", name, "|", e)
                y = np.asarray(x, dtype=float).copy()
                hr = np.nan

            signals[name] = y
            hrs[name] = hr
            tables[cond].loc[i, name + "_HR"] = hr

        # ========================
        # plot all filters
        # ========================
        fig, ax = plt.subplots(6, 3, figsize=(18, 24))
        ax = ax.ravel()

        for k, (name, _) in enumerate(FILTER_COMBOS):
            y = signals[name]
            ax[k].plot(t, normalize_pm1(y))

            hr_val = hrs[name]
            if np.isfinite(hr_val):
                title = f"{name}  HR={hr_val:.2f}"
            else:
                title = f"{name}  HR=nan"

            ax[k].set_title(title)

        for j in range(len(FILTER_COMBOS), len(ax)):
            ax[j].axis("off")

        plt.tight_layout()

        out_png = DATA_DIR / cond / sample / "filter_compare_v2.png"
        fig.savefig(out_png, dpi=200)
        plt.close()

        print("  figure saved")

    tables[cond].to_csv(FILTER_DIR / f"{cond}.csv", index=False)
    print("Finished:", cond)


Processing: AWR_steady
samples: 50
[1/50] sample_0
  figure saved
[2/50] sample_1
  figure saved
[3/50] sample_2
  figure saved
[4/50] sample_3
  figure saved
[5/50] sample_4
  figure saved
[6/50] sample_5
  figure saved
[7/50] sample_6
  figure saved
[8/50] sample_7
  figure saved
[9/50] sample_8
  figure saved
[10/50] sample_9
  figure saved
[11/50] sample_10
  figure saved
[12/50] sample_11
  figure saved
[13/50] sample_12
  figure saved
[14/50] sample_13
  figure saved
[15/50] sample_14
  figure saved
[16/50] sample_15
  figure saved
[17/50] sample_16
  figure saved
[18/50] sample_17
  figure saved
[19/50] sample_18
  figure saved
[20/50] sample_19
  figure saved
[21/50] sample_20
  figure saved
[22/50] sample_21
  figure saved
[23/50] sample_22
  figure saved
[24/50] sample_23
  figure saved
[25/50] sample_24
  figure saved
[26/50] sample_25
  figure saved
[27/50] sample_26
  figure saved
[28/50] sample_27
  figure saved
[29/50] sample_28
  figure saved
[30/50] sample_29
  figure